In [ ]:
import glob, re
import pandas as pd

train_path = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0]
test_path = glob.glob("/kaggle/input/**/test.csv", recursive=True)[0]
tr = pd.read_csv(train_path)
te = pd.read_csv(test_path)

print("train columns:", tr.columns.tolist())
print("test columns:", te.columns.tolist())
print(tr.shape, te.shape)

# 1. Duplicates (normalise: lowercase, drop punctuation)
norm = lambda s: re.sub(r"\W+", " ", str(s).lower()).strip()
tr["n"] = tr["text"].map(norm)
te["n"] = te["text"].map(norm)

print("\nduplicate rows in train:", tr["n"].duplicated().sum())
print("groups with conflicting labels:", (tr.groupby("n")["label"].nunique() > 1).sum())
print("share of test also in train:", te["n"].isin(set(tr["n"])).mean())

# 2. Languages and labels
print("\n", pd.crosstab(tr["language"], tr["label"]))
if "language" in te.columns:
    print("\ntest language mix:\n", te["language"].value_counts(normalize=True))
else:
    print("\nWARNING: test.csv has no language column")

# 3. Text length (characters)
tr["len"] = tr["text"].str.len()
print("\nlength by label:\n", tr.groupby("label")["len"].describe())

In [ ]:
import os, glob, re
import pandas as pd
import kagglehub
from transformers import AutoTokenizer

# 1. Load data
train_path = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0]
tr = pd.read_csv(train_path)

# 2. Find the model folder
# First try the model you added with "Add Input", then fall back to kagglehub
found = [os.path.dirname(f) for f in glob.glob("/kaggle/input/**/config.json", recursive=True)
         if "xlm-roberta-large" in f.lower()]
if found:
    MODEL_DIR = found[0]
else:
    MODEL_DIR = kagglehub.model_download("aisuko/xlm-roberta-large/transformers/default")

print("MODEL_DIR:", MODEL_DIR)
print(os.listdir(MODEL_DIR))

# 3. Token lengths
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
tr["tok"] = [len(x) for x in tok(tr["text"].tolist(), truncation=False)["input_ids"]]

for t in (256, 512):
    print(f"\nshare of rows longer than {t} tokens, by label:")
    print((tr["tok"] > t).groupby(tr["label"]).mean().round(3))

# 4. Read examples
for lang in ["pt", "fr"]:
    for lab in ["PROMPT_INJECTION", "BENIGN"]:
        print(f"\n=== {lang} {lab} ===")
        for t in tr[(tr.language == lang) & (tr.label == lab)].sample(4, random_state=1)["text"]:
            print("-", t[:300].replace("\n", " "))

# 5. Where is the attack in a long PROMPT_INJECTION text?
long_pi = tr[(tr.label == "PROMPT_INJECTION") & (tr.tok > 512)]
print("\n", len(long_pi), "long PROMPT_INJECTION rows")
if len(long_pi):
    t = long_pi.iloc[0]["text"]
    print("START:", t[:400].replace("\n", " "))
    print("\nEND:", t[-400:].replace("\n", " "))

In [ ]:
cues = {"<input>": r"<input>", "[PROMPT]": r"\[PROMPT\]", "PWNED": r"PWNED", "newline": r"\n", "<output>": r"<output>"}
for name, pat in cues.items():
    share = tr["text"].str.contains(pat, regex=True).groupby(tr["label"]).mean().round(3)
    print(name, share.to_dict())

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report

labels = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
y = tr["label"].map({l: i for i, l in enumerate(labels)}).values
strat = (tr["label"] + "_" + tr["language"]).values

X = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3,
                    sublinear_tf=True, max_features=300000).fit_transform(tr["text"])

oof = np.zeros(len(tr), dtype=int)
for f, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(X, strat)):
    clf = LogisticRegression(C=4, max_iter=300, class_weight="balanced").fit(X[a], y[a])
    oof[b] = clf.predict(X[b])
    print("fold", f, "done")

print("macro-F1:", round(f1_score(y, oof, average="macro"), 4))
print(classification_report(y, oof, target_names=labels, digits=3))
print(confusion_matrix(y, oof))
print(pd.Series(oof == y).groupby(tr["language"]).mean().round(3))

In [ ]:
tr["pred"] = [labels[i] for i in oof]
tr["ok"] = tr["pred"] == tr["label"]

# Save for reference (the checker script can use this later)
tr[["id", "label", "pred"]].to_csv("/kaggle/working/tfidf_oof_reference.csv", index=False)

for lang in ["pt", "fr"]:
    sub = tr[tr.language == lang]
    print(f"\n=== {lang}: true (rows) vs predicted (columns) ===")
    print(pd.crosstab(sub.label, sub.pred))

print("\nmean text length by language:")
print(tr.groupby("language")["text"].apply(lambda s: s.str.len().mean()).round(0))

pt_err = tr[(tr.language == "pt") & (~tr.ok)]
for lab, pred in [("BENIGN", "JAILBREAK"), ("JAILBREAK", "BENIGN"), ("PROMPT_INJECTION", "BENIGN")]:
    rows = pt_err[(pt_err.label == lab) & (pt_err.pred == pred)]
    print(f"\n=== pt: true {lab}, predicted {pred}  ({len(rows)} rows) ===")
    for t in rows.sample(min(5, len(rows)), random_state=1)["text"]:
        print("-", t[:250].replace("\n", " "))

In [ ]:
needed = ["tr", "oof", "labels", "pt_err"]
missing = [v for v in needed if v not in globals()]
print("missing:", missing)
print("columns:", [c for c in ["tok", "ok", "pred"] if c in tr.columns])

for t in (256, 384, 512):
    print(f"\nshare longer than {t} tokens, by language:")
    print((tr["tok"] > t).groupby(tr["language"]).mean().round(3))

bins = pd.cut(tr["tok"], [0, 64, 128, 256, 384, 512, 100000],
              labels=["<=64", "65-128", "129-256", "257-384", "385-512", ">512"])
pt = tr[tr.language == "pt"]
print("\nPortuguese: rows per length bucket and label")
print(pd.crosstab(bins[pt.index], pt.label))
print("\nPortuguese: TF-IDF accuracy per length bucket")
print(pt.groupby(bins[pt.index], observed=True)["ok"].mean().round(3))

# Where does the decisive request sit in a long wrongly-predicted BENIGN row?
rows = pt_err[(pt_err.label == "BENIGN") & (pt_err.pred == "JAILBREAK")].sample(3, random_state=2)
for t in rows["text"]:
    print("\nSTART:", t[:150].replace("\n", " "))
    print("END:  ", t[-250:].replace("\n", " "))

In [ ]:
# names the checker cells below use (the floor is the TF-IDF OOF score from above)
LABELS, y_true, TFIDF_FLOOR = labels, y, 0.9155

In [ ]:
def check_submission(path):
    sub = pd.read_csv(path)
    problems = []
    if list(sub.columns) != ["id", "label"]:
        problems.append(f"columns are {list(sub.columns)}, need ['id', 'label']")
    if len(sub) != 6000:
        problems.append(f"{len(sub)} rows, need 6000")
    if sub["id"].duplicated().any():
        problems.append("duplicate ids")
    if set(sub["id"]) != set(te["id"]):
        problems.append("ids do not match test.csv")
    bad = set(sub["label"].unique()) - set(LABELS)
    if bad:
        problems.append(f"invalid labels: {bad}")

    share = sub["label"].value_counts(normalize=True).reindex(LABELS).fillna(0).round(3)
    print("predicted share:", share.to_dict(), "(train is 0.62 / 0.31 / 0.07)")
    # relative check: each class should be within 40% of its train share
    for l, target in zip(LABELS, [0.62, 0.31, 0.07]):
        if not (0.6 * target <= share[l] <= 1.4 * target):
            problems.append(f"{l} share {share[l]} is far from {target}")

    m = te.merge(sub, on="id", how="left")
    print(pd.crosstab(m["language"], m["label"], normalize="index").round(3))
    print("RESULT:", "OK" if not problems else problems)


def check_oof(path):
    p = np.load(path)
    problems = []
    if p.shape != (14000, 3):
        problems.append(f"shape {p.shape}, need (14000, 3)")
    if np.isnan(p).any():
        problems.append("contains NaN")
    if not np.allclose(p.sum(axis=1), 1, atol=1e-2):
        problems.append("rows do not sum to 1 (not softmax probabilities?)")
    if problems:
        print("RESULT:", problems); return
    pred = p.argmax(axis=1)
    f1 = f1_score(y_true, pred, average="macro")
    print("macro-F1:", round(float(f1), 4))
    print("per-class F1:", {l: round(float(v), 3) for l, v in zip(LABELS, f1_score(y_true, pred, average=None))})
    print("per-language accuracy:", pd.Series(pred == y_true).groupby(tr["language"]).mean().round(3).to_dict())
    print(confusion_matrix(y_true, pred))
    if f1 < TFIDF_FLOOR - 0.005:
        problems.append(f"macro-F1 {f1:.4f} is below the TF-IDF floor {TFIDF_FLOOR}: look for a bug")
    print("RESULT:", "OK" if not problems else problems)

In [ ]:
import os

# Test 1: check_oof on the TF-IDF predictions (should show macro-F1 about 0.9155)
onehot = np.eye(3)[oof]
np.save("/kaggle/working/test_oof.npy", onehot)
check_oof("/kaggle/working/test_oof.npy")

# Test 2: check_submission on a deliberately bad file (everything BENIGN)
bad = pd.DataFrame({"id": te["id"], "label": "BENIGN"})
bad.to_csv("/kaggle/working/bad_test.csv", index=False)
check_submission("/kaggle/working/bad_test.csv")

# Clean up the test files
os.remove("/kaggle/working/test_oof.npy")
os.remove("/kaggle/working/bad_test.csv")

In [ ]:
import itertools
from sklearn.model_selection import StratifiedKFold

# weights e^-1.5 .. e^1.5 (0.22 .. 4.5): JAILBREAK can need less than 0.5
def tune_bias(p, y, grid=np.exp(np.arange(-1.5, 1.51, 0.1))):
    # smallest adjustment first, so a tie keeps the weight nearest 1 instead of the grid edge
    pairs = sorted(itertools.product(grid, grid), key=lambda w: abs(np.log(w[0])) + abs(np.log(w[1])))
    best = (-1, 1.0, 1.0)
    for wj, wp in pairs:
        pred = (p * np.array([1.0, wj, wp])).argmax(axis=1)
        f = f1_score(y, pred, average="macro")
        if f > best[0]:
            best = (f, wj, wp)
    return best

# self-check: when no weight helps, the search must leave both weights at 1
assert np.allclose(tune_bias(np.eye(3), np.arange(3))[1:], 1)

def check_bias_stability(path):
    p = np.load(path)
    gains = []
    for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(p, y_true)):
        f_tuned_on_a, wj, wp = tune_bias(p[a], y_true[a])
        raw = f1_score(y_true[b], p[b].argmax(axis=1), average="macro")
        tuned = f1_score(y_true[b], (p[b] * np.array([1.0, wj, wp])).argmax(axis=1), average="macro")
        gains.append(tuned - raw)
        print(f"split {k}: weights JB={wj:.2f} PI={wp:.2f}  raw={raw:.4f}  tuned={tuned:.4f}  gain={tuned-raw:+.4f}")
    print("mean gain on held-out rows:", round(float(np.mean(gains)), 4))

In [ ]:
from sklearn.model_selection import StratifiedKFold

def tune_bias(p, y, grid=np.arange(0.5, 4.01, 0.1)):
    best = (-1, 1.0, 1.0)
    for wj in grid:
        for wp in grid:
            pred = (p * np.array([1.0, wj, wp])).argmax(axis=1)
            f = f1_score(y, pred, average="macro")
            if f > best[0]:
                best = (f, wj, wp)
    return best

def check_bias_stability(path):
    p = np.load(path)
    gains = []
    for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(p, y_true)):
        f_tuned_on_a, wj, wp = tune_bias(p[a], y_true[a])
        raw = f1_score(y_true[b], p[b].argmax(axis=1), average="macro")
        tuned = f1_score(y_true[b], (p[b] * np.array([1.0, wj, wp])).argmax(axis=1), average="macro")
        gains.append(tuned - raw)
        print(f"split {k}: weights JB={wj:.1f} PI={wp:.1f}  raw={raw:.4f}  tuned={tuned:.4f}  gain={tuned-raw:+.4f}")
    print("mean gain on held-out rows:", round(float(np.mean(gains)), 4))

Message to B member
Three requests for train.py, from my data checks:
1. Use max_len 384, not 256. 256 cuts 14% of Portuguese rows, 384 cuts 3%, 512 only 0.9%.
2. Most rows are under 128 tokens, so please use dynamic padding and group_by_length=True. Then 384 costs about the same as 256.
3. Please log the per-language confusion matrix for each fold, not only per-language F1.
Duplicates: keep the 8 duplicate prompts in the same fold. Minor, don't redo finished folds.
When will the first fold's files be ready? I can check them as soon as they exist.

Message to A member
1. Rules check: is a non-pretrained model (TF-IDF char n-grams + LogReg) allowed as an extra blend member? It runs on CPU and I can save OOF and test probabilities in the agreed format. I will only use it if you say yes.
2. Before you apply class-bias weights to the blend, send me the blended OOF. I will test on held-out rows whether the gain is real, so we don't tune on noise.
Note: head+tail truncation helps very few rows (only about 1% of Portuguese rows are over 512 tokens), so a longer max_len is the better lever.
My Kaggle GPU quota is free if you want a spare slot for a seed or fold job.

Message to team
Data check results (from train/test):
- Test has a language column. Same language mix as train (pt 45%, fr 25%, it 20%, en 10%).
- Labels are 62/31/7 in every language, so language prefix and per-language bias are unlikely to help.
- 8 duplicate prompts in train, no conflicting labels. Only about 5 test rows also appear in train.
- English PROMPT_INJECTION has only 98 rows, so per-language PI scores will be noisy.

Baseline: char TF-IDF + LogReg, 5-fold OOF macro-F1 = 0.9155
(BENIGN 0.941, JAILBREAK 0.904, PROMPT_INJECTION 0.902).
Use 0.92 as a sanity floor: any XLM-R fold below it means a bug (label map, row order, collapsed model).

Biggest weakness: Portuguese. Accuracy 0.88 vs 0.96+ for the others, and it is 45% of test.
The errors are BENIGN <-> JAILBREAK (pt: 517 BENIGN predicted JAILBREAK, 210 the other way; fr: 33 and 15).

File checker is ready. Please send me every OOF file (14000x3, train.csv order), test probability file (6000x3) and submission CSV before it is used. I will reply with OK or the exact problem.